# Make Round 2 Parts

Strategy: build genbank parts that can be uploaded into teselagen.

## Start by making the last round's plasmids...

In [ ]:
# Read oligos from round 1
import pandas as pd
round1_oligos = pd.concat([
    pd.read_csv('notebooks/jacob/round1/round1_idt_oligos.csv'),
    pd.read_csv('notebooks/jacob/round1/round1_idt_longoligos.csv')
])
round1_oligos[round1_oligos['Sequence Name'].str.contains('.R1')].head()

,Sales Order,Plate Name,Reference,Manufacturing ID,Product,Purification,Sequence Name,Sequence Notes,Unit Size,Bases,...,Tm (50mM NaCl) C,Modifications and Services,Final OD,nmoles,Conc,Final Volume uL,Buffer,Print Date,Well Position,Volume
12,20999821,round1_idt_oligos,570594455,78640356,500 picomole DNA Plate Oligo,Standard Desalting,AP_OplR.R148A.F1,NaN,0.0005,44,...,74.442021,Standard Desalting,NaN,10然 in 50無IDTE Buffer pH 8.0,10然,50.0,IDTE Buffer pH 8.0,3/18/2025 11:43:39 AM,A13,NaN
31,20999821,round1_idt_oligos,570594474,78640281,500 picomole DNA Plate Oligo,Standard Desalting,AP_OplR.R148.R,NaN,0.0005,25,...,65.638318,Standard Desalting,NaN,10然 in 50無IDTE Buffer pH 8.0,10然,50.0,IDTE Buffer pH 8.0,3/18/2025 11:43:40 AM,B08,NaN
222,20999821,round1_idt_oligos,570594665,78640482,500 picomole DNA Plate Oligo,Standard Desalting,PW_GABAT.R157.R,NaN,0.0005,32,...,59.844010,Standard Desalting,NaN,10然 in 50無IDTE Buffer pH 8.0,10然,50.0,IDTE Buffer pH 8.0,3/18/2025 11:43:53 AM,J07,NaN
13,22603348,NaN,570725894,78635427,100 nmole DNA Oligo,Standard Desalting,PW_GABAT.R157W.F1,NaN,0.1000,68,...,67.004136,Standard Desalting LabReady (Normalized to 1...,NaN,100_M in 380_LIDTE Buffer pH 8.0,100_M,NaN,IDTE Buffer pH 8.0,3/18/2025 4:07:56 PM,NaN,380_L


In [4]:
# Utility functions for creating round 1 plasmids
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqFeature import SeqFeature, FeatureLocation
import difflib
import re
import copy

def apply_metagenic_forward_primer(genbank_path, forward_primer_seq, expected_allele):
    # Load GenBank file
    wt_plasmid = SeqIO.read(genbank_path, "genbank")
    mut_plasmid = copy.deepcopy(wt_plasmid)

    # Extract CDS feature
    cds_feature = next(f for f in wt_plasmid.features if f.type == "CDS")
    cds_seq = cds_feature.extract(wt_plasmid.seq)
    cds_start = int(cds_feature.location.start)

    # Align primer to forward strand
    alignment_index, mismatches = find_best_alignment(str(wt_plasmid.seq), forward_primer_seq)

    if mismatches > 3:
        raise ValueError("Primer does not align with 3 or fewer mismatches.")

    # Apply mismatches to the mutated plasmid
    mut_plasmid.seq = mutate_sequence(str(mut_plasmid.seq), forward_primer_seq, alignment_index)

    # Translate and validate allele
    wt_aa, mut_aa, pos = check_allele_mutation(wt_plasmid, mut_plasmid, cds_feature, expected_allele)

    # Verify allele change
    allele_check = f"{wt_aa}{pos}{mut_aa}"
    if allele_check != expected_allele:
        raise ValueError(f"Expected allele {expected_allele}, but got {allele_check}.")

    # Add annotation for the mutation
    mut_start = cds_feature.location.start + (pos - 1) * 3
    mut_end = mut_start + 3
    mutation_feature = SeqFeature(FeatureLocation(mut_start, mut_end), type="mutation", qualifiers={"label": [expected_allele]})
    mut_plasmid.features.append(mutation_feature)

    return mut_plasmid

def find_best_alignment(template_seq, primer_seq):
    best_index = -1
    best_mismatches = len(primer_seq) + 1
    for i in range(len(template_seq) - len(primer_seq) + 1):
        window = template_seq[i:i + len(primer_seq)]
        mismatches = sum(1 for a, b in zip(window, primer_seq) if a != b)
        if mismatches < best_mismatches:
            best_mismatches = mismatches
            best_index = i
    return best_index, best_mismatches

def mutate_sequence(template_seq, primer_seq, start_idx):
    template_list = list(template_seq)
    for i, base in enumerate(primer_seq):
        if template_list[start_idx + i] != base:
            template_list[start_idx + i] = base
    return Seq("".join(template_list))

def check_allele_mutation(wt_plasmid, mut_plasmid, cds_feature, allele_id):
    # Parse allele ID, e.g., A73Y
    match = re.match(r"([A-Z])(\d+)([A-Z])", allele_id)
    if not match:
        raise ValueError("Allele ID must be in the format A73Y.")

    wt_aa, pos, mut_aa = match.groups()
    pos = int(pos)

    cds_start = int(cds_feature.location.start)
    cds_end = int(cds_feature.location.end)

    wt_cds = wt_plasmid.seq[cds_start:cds_end]
    mut_cds = mut_plasmid.seq[cds_start:cds_end]

    wt_translated = wt_cds.translate()
    mut_translated = mut_cds.translate()

    return wt_translated[pos-1], mut_translated[pos-1], pos

def apply_codon_mutations(genbank_path, allele_id_to_codon):
    """
    Apply mutations to a plasmid using specific codons for each allele.
    
    Parameters
    ----------
    genbank_path : str
        Path to the GenBank file containing the wild-type plasmid
    allele_id_to_codon : dict
        Dictionary mapping allele IDs (e.g., 'A74G') to specific codons (e.g., 'GGG')
    
    Returns
    -------
    Bio.SeqRecord.SeqRecord
        The mutated plasmid with all specified mutations applied
    
    Raises
    ------
    ValueError
        If allele ID format is invalid, codon doesn't encode expected amino acid,
        or wild-type amino acid doesn't match expectation
    """
    from Bio import SeqIO
    from Bio.Seq import Seq
    from Bio.SeqFeature import SeqFeature, FeatureLocation
    import copy
    
    # Load GenBank file
    wt_plasmid = SeqIO.read(genbank_path, "genbank")
    mut_plasmid = copy.deepcopy(wt_plasmid)
    
    # Extract CDS feature
    cds_feature = next(f for f in wt_plasmid.features if f.type == "CDS")
    cds_start = int(cds_feature.location.start)
    
    # Convert plasmid sequence to mutable list
    mut_seq_list = list(str(mut_plasmid.seq))
    
    # Process each mutation
    mutations_applied = []
    for allele_id, codon in allele_id_to_codon.items():
        # Validate codon length and format
        codon = codon.upper()
        if len(codon) != 3 or not all(base in 'ATCG' for base in codon):
            raise ValueError(f"Codon '{codon}' must be exactly 3 DNA bases (A, T, C, G).")
        
        # Verify that the codon encodes the expected amino acid
        codon_aa = str(Seq(codon).translate())
        expected_aa = allele_id[-1]  # Last character of allele ID
        if codon_aa != expected_aa:
            raise ValueError(f"Codon '{codon}' encodes '{codon_aa}', not '{expected_aa}' as expected for allele {allele_id}.")
        
        # Use check_allele_mutation to validate the wild-type state and get position
        wt_aa, _, pos = check_allele_mutation(wt_plasmid, wt_plasmid, cds_feature, allele_id)
        
        # Calculate nucleotide position in the plasmid
        codon_start_pos = cds_start + (pos - 1) * 3
        
        # Apply the mutation
        for i, base in enumerate(codon):
            mut_seq_list[codon_start_pos + i] = base
        
        mutations_applied.append((allele_id, pos, codon_start_pos, codon))
    
    # Update the mutated plasmid sequence
    mut_plasmid.seq = Seq("".join(mut_seq_list))
    
    # Verify all mutations using check_allele_mutation
    for allele_id, _, _, _ in mutations_applied:
        wt_aa, mut_aa, pos = check_allele_mutation(wt_plasmid, mut_plasmid, cds_feature, allele_id)
        allele_check = f"{wt_aa}{pos}{mut_aa}"
        if allele_check != allele_id:
            raise ValueError(f"Failed to apply mutation: expected {allele_id}, got {allele_check}")
    
    # Add mutation annotations
    for allele_id, pos, codon_start_pos, codon in mutations_applied:
        mutation_feature = SeqFeature(
            FeatureLocation(codon_start_pos, codon_start_pos + 3),
            type="mutation",
            qualifiers={"label": [allele_id], "codon": [codon]}
        )
        mut_plasmid.features.append(mutation_feature)
    
    # Update plasmid ID and name to reflect mutations
    mutation_names = "_".join(sorted(allele_id_to_codon.keys()))
    mut_plasmid.id = f"{wt_plasmid.id}_{mutation_names}"
    mut_plasmid.name = f"{wt_plasmid.name}_{mutation_names}"
    mut_plasmid.description = f"{wt_plasmid.description} with mutations: {mutation_names}"
    
    return mut_plasmid

In [101]:
# Create round 1 plasmids in backend/src/notebooks/jacob/round1/new_plasmids
import logging

# Process only forward primers
for _, row in round1_oligos.iterrows():
    primer_name = row["Sequence Name"]
    primer_seq = row["Sequence"].replace(' ','')

    if primer_name.endswith('.R'):
        continue

    parts = primer_name.split('.')
    if len(parts) < 3:
        logging.error(f"Skipping malformed primer name: {primer_name}")
        continue

    campaign_id = parts[0]  # e.g., 'AP_OplR'
    mutation_id = parts[1]  # e.g., 'W10L'

    input_path = f'notebooks/jacob/round1/template_plasmids/{campaign_id}.gb'
    output_id = f"{campaign_id}-{mutation_id}"
    output_path = f'notebooks/jacob/round1/new_plasmids/{output_id}.gb'

    try:
        mut_plasmid = apply_metagenic_forward_primer(input_path, primer_seq, mutation_id)
        mut_plasmid.id = output_id
        mut_plasmid.name = output_id
        SeqIO.write(mut_plasmid, output_path, 'genbank')
        # logging.info(f"Wrote: {output_path}")
    except Exception as e:
        logging.error(f"Error processing {primer_name}: {e}")

/Users/jacobroberts/git/foldy/.venv/lib/python3.12/site-packages/Bio/GenBank/__init__.py:368: BiopythonParserWarning: Non-upper case molecule type in LOCUS line: dna
  warnings.warn(
/Users/jacobroberts/git/foldy/.venv/lib/python3.12/site-packages/Bio/SeqFeature.py:1043: BiopythonParserWarning: Attempting to fix invalid location '5639..386' as it looks like incorrect origin wrapping. Please fix input file, this could have unintended behavior.
  warnings.warn(
/Users/jacobroberts/git/foldy/.venv/lib/python3.12/site-packages/Bio/SeqFeature.py:1043: BiopythonParserWarning: Attempting to fix invalid location '6052..398' as it looks like incorrect origin wrapping. Please fix input file, this could have unintended behavior.
  warnings.warn(
/Users/jacobroberts/git/foldy/.venv/lib/python3.12/site-packages/Bio/SeqIO/InsdcIO.py:404: BiopythonWarning: Feature qualifier key 'ApEinfo_graphicformat' is longer than maximum length specified by standard (20 characters).
  warnings.warn(
/Users/jacobro

### Then make the misc ones...

In [102]:
# Make a couple more misc plasmids that weren't intentional, but were nevertheless relevant.
from app.helpers.sequence_util import allele_set_to_seq_id
misc_mutations = {
    'ID_CUS': [
        {
            'M295V': 'GTG',
            'G223S': 'AGC',
        }
    ]
}
for campaign_id, mutants in misc_mutations.items():
    for codon_dict in mutants:
        mutant_seq_id = allele_set_to_seq_id(set(codon_dict.keys()))
        input_path = f'notebooks/jacob/round1/template_plasmids/{campaign_id}.gb'
        output_id = f"{campaign_id}-{mutant_seq_id}"
        output_path = f'notebooks/jacob/round1/new_plasmids/{output_id}.gb'


        try:
            mut_plasmid = apply_codon_mutations(input_path, codon_dict)
            mut_plasmid.id = output_id
            mut_plasmid.name = output_id
            SeqIO.write(mut_plasmid, output_path, 'genbank')
            # logging.info(f"Wrote: {output_path}")
        except Exception as e:
            logging.error(f"Error processing {campaign_id}-{mutant_seq_id}: {e}")

# Define Round 2 Seq IDs

In [1]:
# Generate random mutations for AP_OplR
import numpy as np

from app.helpers.sequence_util import VALID_AMINO_ACIDS
AP_OplR_sequence = '''MPTELSTHGWPQPERQVRWAEAISDTYFPLSLEFAAGQPFDGHLQRWSTPSTPLSLSRLRSSQLGYSRSKAHIGQDHEAFYLVTVPHGSEVHFEQDGHQISCSPGGFIVERGDAPYRFHYGTQNDLWVLKLPERALKANLLGHKRYTRHCFDARQGLGRIFVEQLDLCARHFDTSPPAARHLLLEQASATLLLALQQDERVLGSEGSNLSTLHLTRVEQYVQQNLGDPELSPQTIAGACGLSLRYLHKLFAITPYTLNEWVRLQRLEAVHRQLRDPHCHLAIGELAFRWGFADQAQFTRAFRQQYGCTASEVRATRTH'''

random_mutations = []
while len(random_mutations) < 24:
    index = np.random.choice(range(len(AP_OplR_sequence)), size=1)[0]
    wt_allele = AP_OplR_sequence[index]
    new_allele = np.random.choice([a for a in VALID_AMINO_ACIDS if a != wt_allele])
    random_mutations.append(f'{wt_allele}{index + 1}{new_allele}')

print('\n'.join(random_mutations))


A268I
C239G
E110Y
Q264N
A292Y
K144T
W289D
A187H
L230I
A189I
T49S
E218K
P53R
T3D
L202G
R117Y
Q233M
R148H
K248R
L129G
F40I
L212M
G105C
L225S


In [2]:
def sort_seq_ids(seq_ids):
    return sorted(seq_ids, key=lambda x: (int(x[1:-1]), x[0]))


SK_Art28_seq_ids = '''A32D_V247E
S239E_R262A
A32D_Y229D
Y100L_Y229D
Y100L_Y229E
Y100K_Y229K
Y100K_Y229G
Y100K_Y229E
Y100K_Y229Q
Y100K_G243D
Y100K_Y229D
Y100K_Y229A'''.split('\n')

LK_BorAT_seq_ids = '''K2715T
N2505G
S2468P
M1323V_M1958V
W1806L_K2715S
M1958V_K2715S
M905R_N2505A
M905T_K2715T
Q1514R_M1958V
W1806V_K2715S
W1806L_M1958V
M905E_K2715T
M905A_K2715T
W1806V_M1958V
W1806A_K2715T
W1806A_M1958V
Q764R_M1323I
W1806A_K2715S
M905E_N2505A
M905G_N2505A
Q764R_M1323V
W1806L_N2505A
M1958L_K2715S
M905R_K2715T
W1806A_N2505A
M905G_K2715T
M1323I_M1958V'''.split('\n')

TY_Pop2_seq_ids = '''Q70K
D186N
K234E
E310R
E360D
F420L
R200K
I182V
D104G_G429R
D104R_G429R
D152E_G429R
F265R_G429R
F61L_F265R
F61L_G429R
G416R_G429R
G429R_T458R
G429R_T473A
H134R_G429R
H8K_G429R
H8Q_G429R
H8R_G429R
K234E_G429R
M320R_G429R
M381Y_G429R
Q184S_G429R
Q237E_G429R
S125P_G429R
S371R_G429R
T9R_G429R
W412L_G429R
Y128M_G429R
Y327A_G429R
Y405A_G429R
Y327P_G429R'''.split('\n')

GAH_DB_seq_ids = '''M417L_L420E
S23T_M105E_S189L
S23T_S189L_C238E
S23T_S189L_C238K
N35K_S76Y_L294P_A378T
S76Y_M105E_L294P_A378T
S76Y_M105K_L294P_A378T
S76Y_M105R_L294P_A378T
S76Y_C238E_L294P_A378T
S76Y_C238K_L294P_A378T
S76Y_C238S_L294P_A378T
S76Y_C238T_L294P_A378T
S76Y_C242K_L294P_A378T
S76Y_C242T_L294P_A378T
S76Y_H273K_L294P_A378T
S76Y_L294P_M308T_A378T
S76Y_L294P_L320K_A378T
S76Y_L294P_A378T_H386E
S76Y_L294P_A378T_H386K
S76Y_L294P_A378T_H386R
S76Y_L294P_A378T_M417V
S76Y_L294P_A378T_L420K
S76Y_L294P_A378T_F427K
S76Y_L294P_A378T_F427S'''.split('\n')

GAH_DBAT_take2_seq_ids = '''S23T_S189L_L420E
S23T_S189L_G361I
G361L_M417L
S23T_S189L_C242K
S76Y_C238K_L294P_A378T
S23T_S189L_G361V
S189L_G361L
C238K_M417L
S76Y_C242K_L294P_A378T
S23T_M105E_S189L
M105E_M417L
S23T_S189L_G361L
S76Y_M105E_L294P_A378T
S76Y_I175Q_L294P_A378T
S76Y_L294P_A378T_L420S
S76Y_L294P_G361V_A378T
S76Y_L294P_A378T_F427E
M105E_S189L
S76Y_L294P_G361I_A378T
G361I_M417L
S23T_S189L_M417L
S23T_S189L_C238K
S76Y_L294P_G361T_A378T
S76Y_L294P_A378T_M417L'''.split('\n')

ID_CUS_seq_ids = '''R271G_H400F
R271G_W375L
F216P_R271G
N264E_R271G
T209R_R271G
G223S_R271G_M295V
R271G_M295A
R271G_M295V
R271G_M323L
R271G_Q365R
T218D_R271G
G223S_M295V_K367E
K121R_G223S_M295V
N142T_G223S_M295V
K121A_R271G
K121A_Q279D
K121A_G223S_M295V
K121G_R271G
K121V_R271G
K121C_R271G
K121R_R271G
H82L_R271G
H82F_R271G
R271G_G274F'''.split('\n')

AP_OplR_seq_ids = list(set('''F80A_Y255R
Y255T_H279K
Y220H_Y255M
Y146C_Y255T
Y255T_E259S
Q75R_Y255T
Y255M_L280R
Y146V_Y255T
Y146I_Y255T
T52G_Y255T
Y146M_Y255T
Y146A_Y255T
Y220H_Y255T
Y255T_Q296R
T215R_Y255L
Y146S_Y255M
Q123D_Y255R
Y146T_Y255T
Y146S_Y255T
Y146L_Y255T
Y255T_Q296H
Y255T_Q296C
Y255M_E259S
Y146I_Y255M
W10V_Q123D
Q123D_V312Y
Q123D_Q155T
Q45R_Q123D
Q123D_A309P
D25R_Q63A
Q123D_L241I
Q63V_Q123D
Q12D_Q123D
Q123D_T190L
P2I_Q123D
Q16A_Q123D
Q63A_Q296H
Q123D_Q296H
Q99R_Q123D
Q123D_H318D
Q63A_E218I
Q45S_Q123D
Q123D_P254G
Q123D_A187I
Q123D_Q264K
Q123D_G237K
Q123D_Q155S
Q123D_Q303R
E218L_Y255T
Y255L_A309P
Y255M_A309P
E218R_Y255L
K137P_Y255L
N139A_Y255L
E218T_Y255T
Y255M_Q296H
K144E_Y255L
E218R_Y255M
K144A_Y255T
Y255T_A309P
I252G_Y255M
K144E_Y255M
K144A_Y255L
K144D_Y255M
E218I_Y255T
E218M_Y255L
P29R_Y255L
E218R_Y255T
K144D_Y255L
Y255L_Q296H
K137A_Y255L
G237R_Y255T'''.split('\n')))

AP_OplR_random_choice_seq_ids = '''R58H
S57Y
P132L
S204A
G226M
A36T
S188P
L263N
R271C
R111Y
W127D
V83F
L266K
R288I
H98M
F287S
H77N
T298S
K248F
P115I
L191F
R299N
G106R
W47C'''.split('\n')

# part_list = []
# for ii, seq_id in enumerate(AP_OplR_seq_ids):
#     parts = split_plasmid_for_mutations(
#         'src/notebooks/jacob/data/plasmids/AP_OplR_WT.gb',
#         AP_OplR_WT,
#          seq_id,
#          prefix='AP_OplR_R2_P',
#          suffix=f'_{ii+1:02}'
#     )
#     part_list.extend(parts)
#     for part in parts:
#         SeqIO.write(part, f'src/notebooks/jacob/data/round2_parts/{part.id}.gb', 'genbank')


In [7]:
# Utility function for finding the 
import logging
from collections import defaultdict
import glob
from app.helpers.sequence_util import allele_set_to_seq_id
from pathlib import Path

def get_seq_id_to_build_tuples(seq_ids, campaign_id, previous_round_plasmid_repo='notebooks/jacob/round1/new_plasmids'):
    base_seq_ids = []
    for fpath in Path(previous_round_plasmid_repo).glob(f'{campaign_id}*.gb'):
        base_seq_ids.append(fpath.stem.split('-')[1])

    possible_base_frequency = defaultdict(int)
    for new_seq_id in seq_ids:
        new_seq_id_allele_set = set(new_seq_id.split('_'))
        for possible_novel_allele in new_seq_id_allele_set:
            required_base_seq_id = allele_set_to_seq_id(new_seq_id_allele_set - {possible_novel_allele})
            if required_base_seq_id in base_seq_ids:
                possible_base_frequency[required_base_seq_id] += 1
    possible_base_frequency = dict(possible_base_frequency)

    seq_id_to_build_tuple = {}
    for new_seq_id in seq_ids:
        new_seq_id_allele_set = set(new_seq_id.split('_'))
        if len(new_seq_id_allele_set) == 1:
            seq_id_to_build_tuple[new_seq_id] = ('WT', list(new_seq_id_allele_set)[0])
            continue

        # Always prioritize more common possible bases.
        for possible_base, frequency in sorted(possible_base_frequency.items(), key=lambda x: x[1], reverse=True):
            possible_base_allele_set = set(possible_base.split('_'))
            if possible_base_allele_set.issubset(new_seq_id_allele_set):
                required_new_allele_set = new_seq_id_allele_set - possible_base_allele_set
                if len(required_new_allele_set) == 1:
                    seq_id_to_build_tuple[new_seq_id] = (possible_base, list(required_new_allele_set)[0])
                    break
        else:
            logging.error(f"No base sequence found for {new_seq_id}. Skipping.")

    seq_id_to_build_tuple = dict(seq_id_to_build_tuple)
    return seq_id_to_build_tuple
get_seq_id_to_build_tuples(ID_CUS_seq_ids, 'ID_CUS')

{'R271G_H400F': ('R271G', 'H400F'),
 'R271G_W375L': ('R271G', 'W375L'),
 'F216P_R271G': ('R271G', 'F216P'),
 'N264E_R271G': ('R271G', 'N264E'),
 'T209R_R271G': ('R271G', 'T209R'),
 'G223S_R271G_M295V': ('G223S_M295V', 'R271G'),
 'R271G_M295A': ('R271G', 'M295A'),
 'R271G_M295V': ('R271G', 'M295V'),
 'R271G_M323L': ('R271G', 'M323L'),
 'R271G_Q365R': ('R271G', 'Q365R'),
 'T218D_R271G': ('R271G', 'T218D'),
 'G223S_M295V_K367E': ('G223S_M295V', 'K367E'),
 'K121R_G223S_M295V': ('G223S_M295V', 'K121R'),
 'N142T_G223S_M295V': ('G223S_M295V', 'N142T'),
 'K121A_R271G': ('R271G', 'K121A'),
 'K121A_Q279D': ('Q279D', 'K121A'),
 'K121A_G223S_M295V': ('G223S_M295V', 'K121A'),
 'K121G_R271G': ('R271G', 'K121G'),
 'K121V_R271G': ('R271G', 'K121V'),
 'K121C_R271G': ('R271G', 'K121C'),
 'K121R_R271G': ('R271G', 'K121R'),
 'H82L_R271G': ('R271G', 'H82L'),
 'H82F_R271G': ('R271G', 'H82F'),
 'R271G_G274F': ('R271G', 'G274F')}

In [4]:
AP_OplR_random_choice_seq_ids

['R58H',
 'S57Y',
 'P132L',
 'S204A',
 'G226M',
 'A36T',
 'S188P',
 'L263N',
 'R271C',
 'R111Y',
 'W127D',
 'V83F',
 'L266K',
 'R288I',
 'H98M',
 'F287S',
 'H77N',
 'T298S',
 'K248F',
 'P115I',
 'L191F',
 'R299N',
 'G106R',
 'W47C']

In [9]:
import pandas as pd
# Create a CSV file with all the target plasmids
from app.helpers.sequence_util import sort_seq_id_list_no_verification

round2_seq_dict_list = []
def add_campaign_to_round2_list(campaign_id, design_id, seq_ids):
    seq_id_to_build_tuples = get_seq_id_to_build_tuples(seq_ids, campaign_id)
    for seq_index, new_seq_id in enumerate(sort_seq_id_list_no_verification(seq_ids)):
        base_seq_id, new_allele_id = seq_id_to_build_tuples[new_seq_id]
        round2_seq_dict_list.append({
            'campaign_id': campaign_id,
            'design_id': design_id,
            'seq_id': new_seq_id,
            'teselagen_plasmid_id': f'{design_id}_{seq_index+1:04}',
            'base_seq_id': base_seq_id,
            'new_allele_id': new_allele_id,
        })

add_campaign_to_round2_list('TY_Pop2', 'TY_Pop2_R2', TY_Pop2_seq_ids)
add_campaign_to_round2_list('LK_BorAT', 'LK_BorAT_R2_retry', LK_BorAT_seq_ids)
add_campaign_to_round2_list('GAH_DBAT', 'GAH_DBAT_R2', GAH_DB_seq_ids)
add_campaign_to_round2_list('GAH_DBAT', 'GAH_DBAT_R2_take2', GAH_DBAT_take2_seq_ids)
add_campaign_to_round2_list('ID_CUS', 'ID_CUS_R2', ID_CUS_seq_ids)
add_campaign_to_round2_list('AP_OplR', 'AP_OplR_R2', AP_OplR_seq_ids)
add_campaign_to_round2_list('AP_OplR', 'AP_OplR_R2_random', AP_OplR_random_choice_seq_ids)
add_campaign_to_round2_list('SK_Art', 'SK_Art_R2', SK_Art28_seq_ids)

round2_seq_df = pd.DataFrame(round2_seq_dict_list)

round2_seq_df.to_excel('notebooks/jacob/round2/250604_round2_seq_ids.xlsx', index=False)

In [10]:
round2_seq_df

,campaign_id,design_id,seq_id,teselagen_plasmid_id,base_seq_id,new_allele_id
0,TY_Pop2,TY_Pop2_R2,Q70K,TY_Pop2_R2_0001,WT,Q70K
1,TY_Pop2,TY_Pop2_R2,I182V,TY_Pop2_R2_0002,WT,I182V
2,TY_Pop2,TY_Pop2_R2,D186N,TY_Pop2_R2_0003,WT,D186N
3,TY_Pop2,TY_Pop2_R2,R200K,TY_Pop2_R2_0004,WT,R200K
4,TY_Pop2,TY_Pop2_R2,K234E,TY_Pop2_R2_0005,WT,K234E
...,...,...,...,...,...,...
236,SK_Art,SK_Art_R2,Y100K_Y229Q,SK_Art_R2_0008,Y100K,Y229Q
237,SK_Art,SK_Art_R2,Y100K_G243D,SK_Art_R2_0009,Y100K,G243D
238,SK_Art,SK_Art_R2,Y100L_Y229D,SK_Art_R2_0010,Y100L,Y229D
239,SK_Art,SK_Art_R2,Y100L_Y229E,SK_Art_R2_0011,Y100L,Y229E


In [11]:
for design_id in round2_seq_df.design_id.unique():
    print(design_id)
    design_round2_seq_df = round2_seq_df[round2_seq_df.design_id == design_id]
    template_counts = design_round2_seq_df.groupby('base_seq_id').seq_id.count().sort_values(ascending=False)
    for base_seq_id, count in template_counts.items():
        print(f'{base_seq_id} (template for {count} new mutants)')
    print()

TY_Pop2_R2
G429R (template for 25 new mutants)
WT (template for 8 new mutants)
F61L (template for 1 new mutants)

LK_BorAT_R2_retry
M1958V (template for 7 new mutants)
K2715T (template for 6 new mutants)
N2505A (template for 5 new mutants)
K2715S (template for 4 new mutants)
WT (template for 3 new mutants)
Q764R (template for 2 new mutants)

GAH_DBAT_R2
S76Y_L294P_A378T (template for 20 new mutants)
S23T_S189L (template for 3 new mutants)
M417L (template for 1 new mutants)

GAH_DBAT_R2_take2
S76Y_L294P_A378T (template for 10 new mutants)
S23T_S189L (template for 8 new mutants)
M417L (template for 4 new mutants)
S189L (template for 2 new mutants)

ID_CUS_R2
R271G (template for 18 new mutants)
G223S_M295V (template for 5 new mutants)
Q279D (template for 1 new mutants)

AP_OplR_R2
Y255T (template for 23 new mutants)
Q123D (template for 22 new mutants)
Y255L (template for 12 new mutants)
Y255M (template for 11 new mutants)
Q63A (template for 2 new mutants)
F80A (template for 1 new mutants)

In [12]:
template_name_list = []
template_well_location_list = []
for campaign_id, row_name in zip(round2_seq_df.campaign_id.unique(), 'ABCDEFGHIJKLMNO'):
    campaign_round2_base_seq_df = round2_seq_df[round2_seq_df.campaign_id == campaign_id]
    template_counts = campaign_round2_base_seq_df.groupby('base_seq_id').seq_id.count().sort_values(ascending=False)
    # for base_seq_id, count in template_counts.items():
    for base_seq_id in sort_seq_id_list_no_verification(campaign_round2_base_seq_df.base_seq_id.unique()):
        if base_seq_id == 'WT':
            template_name_list.append(campaign_id)
        else:
            template_name_list.append(f'{campaign_id}-{base_seq_id}')
    
    for col_num in range(1, template_counts.shape[0] + 1):
        template_well_location_list.append(f'{row_name}{col_num}')
for t in template_name_list:
    print(t)
for t in template_well_location_list:
    print(t)

TY_Pop2
TY_Pop2-F61L
TY_Pop2-G429R
LK_BorAT
LK_BorAT-Q764R
LK_BorAT-M1958V
LK_BorAT-N2505A
LK_BorAT-K2715S
LK_BorAT-K2715T
GAH_DBAT-S189L
GAH_DBAT-M417L
GAH_DBAT-S23T_S189L
GAH_DBAT-S76Y_L294P_A378T
ID_CUS-R271G
ID_CUS-Q279D
ID_CUS-G223S_M295V
AP_OplR
AP_OplR-Q63A
AP_OplR-F80A
AP_OplR-Q123D
AP_OplR-Y255L
AP_OplR-Y255M
AP_OplR-Y255T
AP_OplR-Q296H
SK_Art-A32D
SK_Art-Y100K
SK_Art-Y100L
SK_Art-S239E
A1
A2
A3
B1
B2
B3
B4
B5
B6
C1
C2
C3
C4
D1
D2
D3
E1
E2
E3
E4
E5
E6
E7
E8
F1
F2
F3
F4


In [18]:
# Utility class for building Teselagen design
from pathlib import Path
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqFeature import SeqFeature, FeatureLocation
import re
import copy
import hashlib


############################################################
# Teselagen design data classes
############################################################

class TeselagenDesignPart:
    """A part in a Teselagen construct.

    Parameters
    ----------
    nucleic_acid_seq : str, optional
        A short nucleic‑acid sequence (to be synthesised).
    gb_tuple : tuple(str, int, int), optional
        (genbank_path, start, stop) referencing a slice in an existing plasmid.
    """
    def __init__(self, nucleic_acid_seq=None, gb_tuple=None):
        if (nucleic_acid_seq is None) == (gb_tuple is None):
            raise ValueError("Provide *either* nucleic_acid_seq or gb_tuple, not both.")
        self.nucleic_acid_seq = nucleic_acid_seq
        self.gb_tuple = gb_tuple  # (path,start,stop)

    # convenient fingerprint (hashable key) ----------------------------------
    @property
    def _key(self):
        if self.nucleic_acid_seq is not None:
            return ("seq", self.nucleic_acid_seq)
        path, start, stop = self.gb_tuple
        return ("gb", Path(path).resolve().as_posix(), start, stop)

    def to_dict(self):
        if self.nucleic_acid_seq:
            return {"type": "peptide", "sequence": self.nucleic_acid_seq}
        path, start, stop = self.gb_tuple
        return {"type": "reference", "path": path, "start": start, "stop": stop}

    def __repr__(self):
        if self.nucleic_acid_seq:
            return f"TeselagenDesignPart(nucleic_acid_seq={self.nucleic_acid_seq[:10]}… )"
        return f"TeselagenDesignPart(gb_tuple={self.gb_tuple})"


class TeselagenDesignConstruct:
    """Container for parts belonging to one mutant design."""

    def __init__(self, name):
        self.name = name
        self.parts = []  # list[ TeselagenDesignPart ]

    def add_part(self, part: "TeselagenDesignPart"):
        self.parts.append(part)

    def to_dict(self):
        return {"name": self.name, "parts": [p.to_dict() for p in self.parts]}

    def __repr__(self):
        return f"TeselagenDesignConstruct(name={self.name}, parts={self.parts})"


############################################################
# TeselagenDesignBuilder – builds mutants and can emit JSON
############################################################

class TeselagenDesignBuilder:
    """Accumulates Teselagen constructs and can mutate targets, then export."""

    # Simplified codon table (all available codons per AA)
    CODON_TABLE = {
        "A": ["GCT", "GCC", "GCA", "GCG"],
        "R": ["CGT", "CGC", "CGA", "CGG", "AGA", "AGG"],
        "N": ["AAT", "AAC"],
        "D": ["GAT", "GAC"],
        "C": ["TGT", "TGC"],
        "Q": ["CAA", "CAG"],
        "E": ["GAA", "GAG"],
        "G": ["GGT", "GGC", "GGA", "GGG"],
        "H": ["CAT", "CAC"],
        "I": ["ATT", "ATC", "ATA"],
        "L": ["TTA", "TTG", "CTT", "CTC", "CTA", "CTG"],
        "K": ["AAA", "AAG"],
        "M": ["ATG"],
        "F": ["TTT", "TTC"],
        "P": ["CCT", "CCC", "CCA", "CCG"],
        "S": ["TCT", "TCC", "TCA", "TCG", "AGT", "AGC"],
        "T": ["ACT", "ACC", "ACA", "ACG"],
        "W": ["TGG"],
        "Y": ["TAT", "TAC"],
        "V": ["GTT", "GTC", "GTA", "GTG"],
        "*": ["TAA", "TAG", "TGA"],
    }

    def __init__(self, design_name: str):
        self.constructs: list[TeselagenDesignConstruct] = []
        self.design_name = design_name

    # ------------------------------------------------------------------
    # public API – build a mutant construct
    # ------------------------------------------------------------------

    def build_mutant(self, starting_genbank_fpath: str, allele_id: str) -> TeselagenDesignConstruct:
        """Create a mutant construct and store it internally."""
        record = SeqIO.read(starting_genbank_fpath, "genbank")
        cds_feature = next(f for f in record.features if f.type == "CDS")
        cds_nt = cds_feature.extract(record.seq)
        cds_aa = cds_nt.translate()

        m = re.match(r"([A-Z])(\d+)([A-Z])", allele_id)
        if not m:
            raise ValueError("Allele ID must look like A63Y")
        wt_ltr, pos_str, mut_ltr = m.groups()
        pos = int(pos_str)

        if cds_aa[pos - 1] != wt_ltr:
            raise ValueError(
                f"Expected {wt_ltr} at AA pos {pos} in {starting_genbank_fpath}, found {cds_aa[pos-1]}"
            )

        current_codon = str(cds_nt[(pos - 1) * 3 : pos * 3])
        new_codon = self._choose_codon(current_codon, mut_ltr, cds_nt, cds_aa)

        # ------- build part list: upstream / mutant / downstream ----------
        cds_start = int(cds_feature.location.start)
        codon_start = cds_start + (pos - 1) * 3
        codon_end = codon_start + 3

        construct_name = f"{Path(starting_genbank_fpath).stem}_{allele_id}"
        construct = TeselagenDesignConstruct(construct_name)

        if codon_start > 0:
            construct.add_part(
                TeselagenDesignPart(gb_tuple=(starting_genbank_fpath, 0, codon_start))
            )
        construct.add_part(TeselagenDesignPart(nucleic_acid_seq=new_codon))
        if codon_end < len(record):
            construct.add_part(
                TeselagenDesignPart(
                    gb_tuple=(starting_genbank_fpath, codon_end, len(record))
                )
            )

        self.constructs.append(construct)
        return construct

    # ------------------------------------------------------------------
    # helper
    # ------------------------------------------------------------------

    def _choose_codon(self, current_codon: str, aa_letter: str, cds_nt: Seq, cds_aa: Seq) -> str:
        aa_codon_counts = {}
        for aa_idx, aa in enumerate(cds_aa):
            if aa == aa_letter:
                codon = cds_nt[aa_idx * 3: (aa_idx + 1) * 3]
                aa_codon_counts[codon] = aa_codon_counts.get(codon, 0) + 1
        if len(aa_codon_counts) == 0:
            raise ValueError(f"No codons found for {aa_letter} in existing gene ({cds_aa})")
        
        codon_counts_sorted = sorted(aa_codon_counts.items(), key=lambda x: x[1], reverse=True)
        most_common_codon = codon_counts_sorted[0][0]
        if most_common_codon.translate() != aa_letter:
            raise ValueError(f"Bug! Most common codon for {aa_letter} in existing gene ({cds_aa}) is {most_common_codon}, which translates to {most_common_codon.translate()}")
        return str(most_common_codon)

    # ------------------------------------------------------------------
    # Teselagen JSON export
    # ------------------------------------------------------------------

    # ------------------------------------------------------------------
    # Export to Teselagen JSON
    # ------------------------------------------------------------------
    def to_teselagen_json(self, assembly_method="golden gate", allow_duplicates=False):
        """Produce Teselagen‑compatible JSON including *all* annotated GenBank parts."""

        part_key_to_id: dict[tuple, str] = {}
        gb_subsets: dict[str, set[tuple[int, int]]] = {}
        nucleic_parts: dict[str, str] = {}

        def _sha_id(txt: str) -> str:
            # return "p_" + hashlib.sha1(txt.encode()).hexdigest()[:8]
            return f'p_{txt}'
        # ---------------- aggregate parts from constructs ----------------
        for cons in self.constructs:
            for part in cons.parts:
                key = part._key
                if key in part_key_to_id:
                    continue
                if key[0] == "gb":
                    _, pth, s, e = key
                    pid = f"{Path(pth).stem}-{s}-{e}"
                    part_key_to_id[key] = pid
                    gb_subsets.setdefault(pth, set()).add((s, e))
                else:  # synthetic sequence
                    _, seq = key
                    pid = _sha_id(seq)
                    part_key_to_id[key] = pid
                    nucleic_parts[pid] = seq

        # ---------------- sequences array ----------------
        sequences_json = []
        for gb_path, subset_set in gb_subsets.items():
            rec = SeqIO.read(gb_path, "genbank")
            seq_name = Path(gb_path).stem
            parts_json = []

            # 2a) add *all* annotated features as parts
            for feat in rec.features:
                if feat.type == "source":
                    continue
                start = int(feat.location.start)
                end = int(feat.location.end) - 1  # Inclusive for Teselagen
                fid = f"{seq_name}_{start}_{end}_{feat.type}"
                fname = feat.qualifiers.get("label", [feat.type])[0]
                parts_json.append({
                    "start": start,
                    "end": end,
                    "id": fid,
                    "name": fname,
                    "strand": 1 if feat.location.strand != -1 else -1,
                })

            # 2b) add subset slices actually referenced by constructs
            for (s, e) in sorted(subset_set):
                pid = part_key_to_id[("gb", Path(gb_path).resolve().as_posix(), s, e)]
                parts_json.append({
                    "start": s,
                    "end": e - 1,
                    "id": pid,
                    "name": pid,
                    "strand": 1,
                })

            sequences_json.append({
                "name": seq_name,
                "sequence": str(rec.seq),
                "parts": parts_json,
                "circular": True,
            })

        # synthetic sequences
        for pid, seq in nucleic_parts.items():
            sequences_json.append({
                "name": pid,
                "sequence": seq,
                "parts": [{"start": 0, "end": len(seq) - 1, "id": pid, "name": pid, "strand": 1}],
            })


        # --------------------------------------------------
        # 3) Build columns (one per construct)
        # --------------------------------------------------
        columns_json = []
        for column_idx in range(max(len(construct.parts) for construct in self.constructs)):
            col_parts = []
            for construct in self.constructs:
                if column_idx < len(construct.parts):
                    part = construct.parts[column_idx]
                    pid = part_key_to_id[part._key]
                    col_parts.append({"id": pid})
                else:
                    col_parts.append({"id": ""})
            columns_json.append({
                "direction": "forward",
                "icon": "cds",
                "name": f'Part {column_idx + 1}',
                "parts": col_parts,
            })

        # --------------------------------------------------
        # 4) Final JSON structure
        # --------------------------------------------------
        design_json = {
            "assembly_method": assembly_method,
            "columns": columns_json,
            "layout_type": "list",
            "name": self.design_name,
            "sequences": sequences_json,
            # "lab": "25e3cb66-456b-4abd-b52b-52546206955d"
        }

        return {
            "allowDuplicates": allow_duplicates,
            "designJson": design_json,
        }



## Create new plasmids with the Builder

In [19]:
# Populate a builder class

CAMPAIGN_ID = 'TY_Pop2'
DESIGN_ID = f'{CAMPAIGN_ID}_R2'

design_round2_subset_df = round2_seq_df[round2_seq_df.design_id == DESIGN_ID]

builder = TeselagenDesignBuilder(DESIGN_ID)
for row_idx, row in design_round2_subset_df.iterrows():
    seq_id = row['seq_id']
    base_seq_id = row['base_seq_id']
    new_allele = row['new_allele_id']

    if base_seq_id == 'WT':
        genbank_fpath = f'notebooks/jacob/round1/template_plasmids/{CAMPAIGN_ID}.gb'
    else:
        genbank_fpath = f'notebooks/jacob/round1/new_plasmids/{CAMPAIGN_ID}-{base_seq_id}.gb'
    builder.build_mutant(genbank_fpath, new_allele)
builder.to_teselagen_json()

{'allowDuplicates': False,
 'designJson': {'assembly_method': 'golden gate',
  'columns': [{'direction': 'forward',
    'icon': 'cds',
    'name': 'Part 1',
    'parts': [{'id': 'TY_Pop2-0-9023'},
     {'id': 'TY_Pop2-0-9359'},
     {'id': 'TY_Pop2-0-9371'},
     {'id': 'TY_Pop2-0-9413'},
     {'id': 'TY_Pop2-0-9515'},
     {'id': 'TY_Pop2-0-9743'},
     {'id': 'TY_Pop2-0-9893'},
     {'id': 'TY_Pop2-0-10073'},
     {'id': 'TY_Pop2-G429R-0-8837'},
     {'id': 'TY_Pop2-G429R-0-8837'},
     {'id': 'TY_Pop2-G429R-0-8837'},
     {'id': 'TY_Pop2-G429R-0-8840'},
     {'id': 'TY_Pop2-F61L-0-9608'},
     {'id': 'TY_Pop2-G429R-0-8996'},
     {'id': 'TY_Pop2-G429R-0-9125'},
     {'id': 'TY_Pop2-G429R-0-9125'},
     {'id': 'TY_Pop2-G429R-0-9188'},
     {'id': 'TY_Pop2-G429R-0-9197'},
     {'id': 'TY_Pop2-G429R-0-9215'},
     {'id': 'TY_Pop2-G429R-0-9269'},
     {'id': 'TY_Pop2-G429R-0-9365'},
     {'id': 'TY_Pop2-G429R-0-9515'},
     {'id': 'TY_Pop2-G429R-0-9524'},
     {'id': 'TY_Pop2-G429R-0-96

In [20]:
import requests
design = builder.to_teselagen_json()

# This script is used to post a design to the Teselagen API.
def post_design(session, design):
    """Fetch notebook entry by id."""
    url = f"{BASE_URL}/designs"
    response = session.post(url, json=design)

    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return []


BASE_URL = "https://jbei.teselagen.com/tg-api"
USERNAME = "jbr@lbl.gov"  # Replace this with your username
PASSWORD = "5dd21974-c17d-4edd-900f-30630d8c3be9"  # Replace this with your OTP password that you get from the app

session: requests.Session = requests.Session()
session.headers.update(
    {"Content-Type": "application/json", "Accept": "application/json"}
)

# Authenticate and get the token
response: requests.Response = session.put(
    url=f"{BASE_URL}/public/auth",
    json={
        "username": USERNAME,
        "password": PASSWORD,
        "expiresIn": "1d",
    },
)
response.raise_for_status()  # Raise an error if a problem is found
session.headers.update(
    {"x-tg-api-token": response.json()["token"]},  # TOKEN
)
session.headers.pop("Content-Type", None)
del response

# get the example design
# example1 = get_design_json_example1()

# post the design
post_design(session, design)


/Users/jacobroberts/git/foldy/.venv/lib/python3.12/site-packages/Bio/SeqFeature.py:1043: BiopythonParserWarning: Attempting to fix invalid location '14883..480' as it looks like incorrect origin wrapping. Please fix input file, this could have unintended behavior.
  warnings.warn(


Error: 500 - {"code":500,"error":"Could not find part Factor Xa site in sequence with hash NCDe1169dc64889e0ee42d998358296b71b25143803048ebe8b4384211e465483d2.","stack":"InternalServerError: Could not find part Factor Xa site in sequence with hash NCDe1169dc64889e0ee42d998358296b71b25143803048ebe8b4384211e465483d2.\n    at handleUserMessageError (/tg-api/src/utils/handleHttpError.ts:47:12)\n    at Controller.<anonymous> (/tg-api/src/api/modules/b/DesignController/index.ts:472:37)\n    at Generator.throw (<anonymous>)\n    at rejected (/tg-api/build/tg-api/src/api/modules/b/DesignController/index.js:20:65)\n    at runMicrotasks (<anonymous>)\n    at processTicksAndRejections (node:internal/process/task_queues:96:5)"}


[]